In [1]:
import mediapipe as mp
import cv2

In [2]:
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [24]:
pose_model = "/Users/anurag_77y/anaconda_projects/pose_landmarker_full.task"
hand_model="/Users/anurag_77y/anaconda_projects/hand_landmarker.task"
face_model="/Users/anurag_77y/anaconda_projects/face_landmarker.task"

In [4]:
BaseOptions=python.BaseOptions

In [5]:
pose_options = vision.PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=pose_model),
    running_mode=vision.RunningMode.VIDEO
)
pose_detector = vision.PoseLandmarker.create_from_options(pose_options)


I0000 00:00:1776510203.467842  797722 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1776510203.515790  797725 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776510203.525633  797725 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [6]:
hand_options = vision.HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=hand_model),
    running_mode=vision.RunningMode.VIDEO,
    num_hands=2
)
hand_detector = vision.HandLandmarker.create_from_options(hand_options)

I0000 00:00:1776510203.532799  797735 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2
W0000 00:00:1776510203.536641  797737 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776510203.540141  797739 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [7]:
face_options = vision.FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=face_model),
    running_mode=vision.RunningMode.VIDEO
)
face_detector = vision.FaceLandmarker.create_from_options(face_options)

W0000 00:00:1776510203.544160  797744 face_landmarker_graph.cc:180] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
I0000 00:00:1776510203.547480  797744 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2
W0000 00:00:1776510203.548461  797745 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776510203.555825  797745 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [8]:
POSE_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,7),
    (0,4),(4,5),(5,6),(6,8),
    (9,10),
    (11,12),
    (11,13),(13,15),
    (12,14),(14,16),
    (15,17),(16,18),
    (11,23),(12,24),
    (23,24),
    (23,25),(25,27),
    (24,26),(26,28),
    (27,29),(28,30),
    (29,31),(30,32)
]
FACE_POINTS = [
    1,   # nose
    33, 133,   # left eye
    362, 263,  # right eye
    61, 291,   # mouth corners
    13, 14,    # lips center
    70, 300    # eyebrows
]

def draw_pose_skeleton(frame, landmarks):
    h, w, _ = frame.shape

    # Convert all landmarks to pixel coords
    points = []
    for lm in landmarks:
        x = int(lm.x * w)
        y = int(lm.y * h)
        points.append((x, y))

    # Draw lines (bones)
    for connection in POSE_CONNECTIONS:
        start_idx, end_idx = connection
        cv2.line(frame, points[start_idx], points[end_idx], (0,255,0), 2)

    # Draw joints
    for point in points:
        cv2.circle(frame, point, 3, (0,0,255), -1)

In [9]:
# cap=cv2.VideoCapture(0)
# frame_timestamp=0
# while cap.isOpened():
#     ret,frame=cap.read()
#     if not ret:
#         break
#     rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#     mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
#     #run detections
#     pose_result=pose_detector.detect_for_video(mp_image,frame_timestamp)
#     hand_result=hand_detector.detect_for_video(mp_image,frame_timestamp)
#     face_result=face_detector.detect_for_video(mp_image,frame_timestamp)
#     frame_timestamp+=1
#     if hand_result.hand_landmarks:
#         for hand in hand_result.hand_landmarks:
#             for landmark in hand:
#                 x = int(landmark.x * frame.shape[1])
#                 y = int(landmark.y * frame.shape[0])
#                 cv2.circle(frame, (x, y), 5, (255,0,0), -1)

#     # Face
#     if face_result.face_landmarks:
#         for face in face_result.face_landmarks:
#             for landmark in face:
#                 x = int(landmark.x * frame.shape[1])
#                 y = int(landmark.y * frame.shape[0])
#                 cv2.circle(frame, (x, y), 3, (0,0,255), -1)
#     if pose_result.pose_landmarks:
#         draw_pose_skeleton(frame, pose_result.pose_landmarks[0])

#     # Show
#     cv2.imshow("Modern MediaPipe", frame)

#     if cv2.waitKey(1) & 0xFF == ord('q'):
#         break
# cap.release()
# # cv2.destroyAllWindows()
# import csv
# import os

# # If file already exists → remove it (fresh start)
# if os.path.exists("coords.csv"):
#     os.remove("coords.csv")

# headers = ['class']

# # -------- POSE (33 landmarks × 4 values) --------
# for i in range(33):
#     headers += [
#         f'pose_x{i}',
#         f'pose_y{i}',
#         f'pose_z{i}',
#         f'pose_v{i}'
#     ]

# # -------- FACE (reduced points) --------
# for i in range(len(FACE_POINTS)):
#     headers += [
#         f'face_x{i}',
#         f'face_y{i}',
#         f'face_z{i}'
#     ]

# # -------- WRITE HEADER --------
# with open('coords.csv', mode='w', newline='') as f:
#     writer = csv.writer(f)
#     writer.writerow(headers)

# print("New CSV created with headers!")
    


In [10]:
# h, w, _ = frame.shape

# # Pose
# if pose_result.pose_landmarks:
#     for i, lm in enumerate(pose_result.pose_landmarks[0]):
#         x = int(lm.x * w)
#         y = int(lm.y * h)
#         print(f"Pose {i}: ({x},{y}), visibility={lm.visibility:.2f}")

# # Face
# if face_result.face_landmarks:
#     for face in face_result.face_landmarks:
#         for i, lm in enumerate(face):
#             x = int(lm.x * w)
#             y = int(lm.y * h)
#             print(f"Face {i}: ({x},{y})")

In [11]:
import numpy as np

In [12]:
import csv
import os

In [13]:
# num_pose = len(pose_result.pose_landmarks[0]) if pose_result.pose_landmarks else 33
# num_face = len(face_result.face_landmarks[0]) if face_result.face_landmarks else 468


In [14]:
# landmarks=['class']
# for i in range(1, num_pose + 1):
#     landmarks += [f'pose_x{i}', f'pose_y{i}', f'pose_z{i}', f'pose_v{i}']
# for i in range(1, num_face + 1):
#     landmarks += [f'face_x{i}', f'face_y{i}', f'face_z{i}']

In [15]:
# landmarks

In [16]:
# import csv
# import os
# with open('coords.csv', mode='w', newline='') as f:
#     csv_writer = csv.writer(f)
#     csv_writer.writerow(landmarks)

In [17]:
import time
FACE_CONNECTIONS = [
    (33,133), (362,263),   # eyes
    (61,13), (13,14), (14,291),  # mouth
    (70,300)  # eyebrows
]
# ------------------ WEBCAM ------------------

cap = cv2.VideoCapture(0)

start_time = time.time()
prev_timestamp = 0

class_name = None

key_map = {
    ord('h'): 'happy',
    ord('s'): 'sad',
    ord('v'): 'victory',
    ord('n'): 'neutral'
}

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

    # -------- TIMESTAMP FIX --------
    timestamp = int((time.time() - start_time) * 1000)
    if timestamp <= prev_timestamp:
        timestamp = prev_timestamp + 1
    prev_timestamp = timestamp

    # -------- DETECTION --------
    pose_result = pose_detector.detect_for_video(mp_image, timestamp)
    face_result = face_detector.detect_for_video(mp_image, timestamp)
    hand_result=hand_detector.detect_for_video(mp_image,timestamp)
    if hand_result.hand_landmarks:
        for hand in hand_result.hand_landmarks:
            for landmark in hand:
                x = int(landmark.x * frame.shape[1])
                y = int(landmark.y * frame.shape[0])
                cv2.circle(frame, (x, y), 5, (255,0,0), -1)

    # Face
    # if face_result.face_landmarks:
    #     for face in face_result.face_landmarks:
    #         for landmark in face:
    #             x = int(landmark.x * frame.shape[1])
    #             y = int(landmark.y * frame.shape[0])
    #             cv2.circle(frame, (x, y), 3, (0,0,255), -1)
    

    
    if pose_result.pose_landmarks:
        draw_pose_skeleton(frame, pose_result.pose_landmarks[0])

    # -------- KEY INPUT --------
    key = cv2.waitKey(1) & 0xFF

    if key in key_map:
        class_name = key_map[key]
        print("Recording:", class_name)

    if key == ord('q'):
        break

    # -------- DATA COLLECTION --------
    if class_name is not None:
        row = []

        # Pose
        if pose_result.pose_landmarks:
            pose = pose_result.pose_landmarks[0]
            pose_row = np.array([[lm.x, lm.y, lm.z, lm.visibility] for lm in pose]).flatten()
        else:
            pose_row = np.zeros(33 * 4)

        # Face
        if face_result.face_landmarks:
            face = face_result.face_landmarks[0]
            nose=face[1]
            face_row=[]
            for idx in FACE_POINTS:
                for (p1, p2) in FACE_CONNECTIONS:
                    x1 = int(face[p1].x * frame.shape[1])
                    y1 = int(face[p1].y * frame.shape[0])

                    x2 = int(face[p2].x * frame.shape[1])
                    y2 = int(face[p2].y * frame.shape[0])

                    cv2.line(frame, (x1, y1), (x2, y2), (255,0,0), 2)
                lm=face[idx]
                face_row += [
                    lm.x - nose.x,
                    lm.y - nose.y,
                    lm.z
                ]
            face_row = np.array(face_row)
            
        else:
            face_row = np.zeros(len(FACE_POINTS) * 3)

        pose_row = np.nan_to_num(pose_row)
        face_row = np.nan_to_num(face_row)


        row = np.concatenate([pose_row, face_row]).tolist()
        row.insert(0, class_name)

        # Save
        with open('coords.csv', mode='a', newline='') as f:
            csv_writer = csv.writer(f)
            csv_writer.writerow(row)

        cv2.putText(frame, f"Recording: {class_name}", (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

    cv2.imshow("Data Collection", frame)

cap.release()
cv2.destroyAllWindows()


    

W0000 00:00:1776510205.058645  797740 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


Recording: happy
Recording: sad
Recording: victory
Recording: neutral


In [2]:
import pandas as pd

In [3]:
df=pd.read_csv("coords.csv")


df = pd.read_csv("coords.csv")
df = df.dropna()

In [4]:
df.head()

,class,pose_x0,pose_y0,pose_z0,pose_v0,pose_x1,pose_y1,pose_z1,pose_v1,pose_x2,...,face_z7,face_x8,face_y8,face_z8,face_x9,face_y9,face_z9,face_x10,face_y10,face_z10
0,happy,0.551829,0.520163,-0.501935,0.999995,0.569997,0.465609,-0.462517,0.999990,0.582962,...,-0.017192,0.000525,0.062892,-0.016735,-0.072083,-0.090819,0.014982,0.061660,-0.090276,0.022502
1,happy,0.555300,0.518584,-0.526570,0.999995,0.573089,0.463874,-0.490459,0.999990,0.585815,...,-0.017216,-0.000036,0.063545,-0.016767,-0.072689,-0.091201,0.014800,0.061983,-0.089711,0.022676
2,happy,0.557810,0.515519,-0.517475,0.999995,0.574937,0.458900,-0.482051,0.999990,0.587548,...,-0.017224,-0.000394,0.063099,-0.016766,-0.073302,-0.092332,0.014688,0.062156,-0.090321,0.022691
3,happy,0.557994,0.514791,-0.513986,0.999996,0.575743,0.458091,-0.478366,0.999991,0.588829,...,-0.017231,-0.001188,0.064157,-0.016776,-0.073670,-0.092713,0.014402,0.061728,-0.089483,0.022958
4,happy,0.566727,0.522197,-0.532667,0.999994,0.582704,0.461348,-0.498154,0.999988,0.594907,...,-0.017575,-0.001162,0.070741,-0.016413,-0.074236,-0.093406,0.013560,0.061971,-0.089466,0.023471


In [5]:
df.sample(10)

,class,pose_x0,pose_y0,pose_z0,pose_v0,pose_x1,pose_y1,pose_z1,pose_v1,pose_x2,...,face_z7,face_x8,face_y8,face_z8,face_x9,face_y9,face_z9,face_x10,face_y10,face_z10
74,sad,0.628610,0.583031,-0.559970,0.999979,0.632339,0.520959,-0.519900,0.999950,0.638325,...,-0.015286,-0.011538,0.055717,-0.014245,-0.081528,-0.102419,-0.015390,0.025702,-0.083295,0.047669
131,victory,0.569249,0.586339,-0.530380,0.999990,0.589220,0.527203,-0.467603,0.999977,0.601094,...,-0.021434,0.000643,0.056220,-0.020548,-0.065073,-0.075976,0.022802,0.053298,-0.079631,0.031806
19,happy,0.560975,0.520973,-0.573940,0.999978,0.582756,0.462600,-0.539943,0.999954,0.596601,...,-0.019660,-0.000002,0.070742,-0.014915,-0.068167,-0.091825,0.018051,0.061628,-0.089764,0.019864
37,happy,0.652238,0.513937,-0.419104,0.999949,0.644022,0.455608,-0.378575,0.999894,0.647144,...,-0.018841,-0.018341,0.071110,-0.016767,-0.086875,-0.106570,-0.022035,-0.011771,-0.064602,0.065696
139,victory,0.529103,0.613754,-0.587818,0.999993,0.549001,0.553180,-0.539478,0.999986,0.563046,...,-0.017666,0.005063,0.111143,-0.008169,-0.068892,-0.070741,0.017786,0.052175,-0.094983,0.009264
183,neutral,0.556838,0.545269,-0.507154,0.999990,0.574059,0.491454,-0.473062,0.999978,0.585793,...,-0.016071,-0.001566,0.055271,-0.016106,-0.067362,-0.090255,0.014916,0.059703,-0.083517,0.021337
78,sad,0.648972,0.595514,-0.524175,0.999980,0.653457,0.533505,-0.483998,0.999956,0.659766,...,-0.015012,-0.016501,0.053831,-0.013968,-0.078781,-0.117525,-0.021021,0.024784,-0.085092,0.051324
216,neutral,0.518756,0.555978,-0.481952,0.999993,0.545184,0.495959,-0.457806,0.999985,0.561194,...,-0.017716,0.005995,0.059782,-0.017378,-0.046621,-0.079879,0.038018,0.073277,-0.092230,0.002445
214,neutral,0.542045,0.556459,-0.490740,0.999993,0.565483,0.500842,-0.461015,0.999984,0.579801,...,-0.017388,0.001480,0.059164,-0.016738,-0.059131,-0.087289,0.026488,0.068487,-0.088881,0.012566
210,neutral,0.582822,0.564105,-0.510406,0.999992,0.600697,0.509549,-0.476683,0.999984,0.613787,...,-0.017474,-0.003394,0.059303,-0.016687,-0.070245,-0.098005,0.009678,0.057609,-0.084909,0.026640


In [6]:
df[df['class']=='sad']

,class,pose_x0,pose_y0,pose_z0,pose_v0,pose_x1,pose_y1,pose_z1,pose_v1,pose_x2,...,face_z7,face_x8,face_y8,face_z8,face_x9,face_y9,face_z9,face_x10,face_y10,face_z10
50,sad,0.539821,0.481665,-0.500658,0.999981,0.563209,0.438338,-0.459632,0.999960,0.576833,...,-0.026078,0.001239,0.062077,-0.026596,-0.056007,-0.059418,0.040703,0.066939,-0.064640,0.025664
51,sad,0.535410,0.481718,-0.512859,0.999982,0.559678,0.438329,-0.473731,0.999962,0.573451,...,-0.025518,0.002081,0.061920,-0.025490,-0.057223,-0.059520,0.038052,0.066565,-0.066416,0.024262
52,sad,0.532208,0.483727,-0.510370,0.999983,0.557715,0.438980,-0.473394,0.999965,0.571786,...,-0.024678,0.002915,0.061965,-0.024520,-0.058608,-0.061313,0.036495,0.065441,-0.071570,0.022896
53,sad,0.528119,0.483941,-0.501437,0.999985,0.553280,0.438833,-0.468122,0.999968,0.567764,...,-0.024126,0.004512,0.060613,-0.023745,-0.055343,-0.062513,0.036795,0.068062,-0.079653,0.021372
54,sad,0.520395,0.531154,-0.501161,0.999985,0.546314,0.469037,-0.470826,0.999969,0.562135,...,-0.018192,0.006749,0.055745,-0.017407,-0.050220,-0.073723,0.036254,0.068878,-0.102458,0.005866
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108,sad,0.625190,0.591420,-0.575336,0.999982,0.632322,0.529112,-0.534526,0.999960,0.639522,...,-0.014064,-0.010653,0.055009,-0.013139,-0.078639,-0.111227,-0.008716,0.044075,-0.086441,0.036799
109,sad,0.622424,0.590064,-0.579746,0.999982,0.633039,0.528914,-0.536704,0.999960,0.641065,...,-0.014077,-0.010102,0.056339,-0.013147,-0.077602,-0.112897,-0.007482,0.045532,-0.085814,0.035425
110,sad,0.620461,0.589976,-0.579391,0.999983,0.631667,0.528934,-0.535441,0.999961,0.639934,...,-0.014074,-0.009823,0.055726,-0.013092,-0.075572,-0.114552,-0.006791,0.047974,-0.086519,0.034836
111,sad,0.619402,0.592188,-0.547201,0.999984,0.631360,0.531656,-0.507898,0.999964,0.640212,...,-0.014418,-0.009336,0.056594,-0.013302,-0.075223,-0.114676,-0.004506,0.049696,-0.084684,0.034637


In [7]:
df.shape


(238, 166)

In [8]:
from sklearn.model_selection import train_test_split

In [9]:
X = df.drop('class', axis=1) # features
y = df['class'] # target value

In [10]:
X

,pose_x0,pose_y0,pose_z0,pose_v0,pose_x1,pose_y1,pose_z1,pose_v1,pose_x2,pose_y2,...,face_z7,face_x8,face_y8,face_z8,face_x9,face_y9,face_z9,face_x10,face_y10,face_z10
0,0.551829,0.520163,-0.501935,0.999995,0.569997,0.465609,-0.462517,0.999990,0.582962,0.468094,...,-0.017192,0.000525,0.062892,-0.016735,-0.072083,-0.090819,0.014982,0.061660,-0.090276,0.022502
1,0.555300,0.518584,-0.526570,0.999995,0.573089,0.463874,-0.490459,0.999990,0.585815,0.466554,...,-0.017216,-0.000036,0.063545,-0.016767,-0.072689,-0.091201,0.014800,0.061983,-0.089711,0.022676
2,0.557810,0.515519,-0.517475,0.999995,0.574937,0.458900,-0.482051,0.999990,0.587548,0.461242,...,-0.017224,-0.000394,0.063099,-0.016766,-0.073302,-0.092332,0.014688,0.062156,-0.090321,0.022691
3,0.557994,0.514791,-0.513986,0.999996,0.575743,0.458091,-0.478366,0.999991,0.588829,0.461036,...,-0.017231,-0.001188,0.064157,-0.016776,-0.073670,-0.092713,0.014402,0.061728,-0.089483,0.022958
4,0.566727,0.522197,-0.532667,0.999994,0.582704,0.461348,-0.498154,0.999988,0.594907,0.464309,...,-0.017575,-0.001162,0.070741,-0.016413,-0.074236,-0.093406,0.013560,0.061971,-0.089466,0.023471
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
233,0.541088,0.555631,-0.552142,0.999969,0.560977,0.495711,-0.512012,0.999909,0.574307,0.495822,...,-0.016517,0.001427,0.058261,-0.015940,-0.069933,-0.080590,0.016272,0.058357,-0.095098,0.019404
234,0.544944,0.555440,-0.536516,0.999972,0.564539,0.495632,-0.499662,0.999917,0.577246,0.495773,...,-0.016390,0.000762,0.058481,-0.015779,-0.070499,-0.082098,0.014493,0.057884,-0.092642,0.020435
235,0.550039,0.555585,-0.550018,0.999974,0.568729,0.496305,-0.515686,0.999923,0.581293,0.496624,...,-0.016393,0.000247,0.058939,-0.015788,-0.071056,-0.083069,0.014125,0.057508,-0.090614,0.021664
236,0.557886,0.556041,-0.503946,0.999976,0.574761,0.498222,-0.471557,0.999930,0.586870,0.498856,...,-0.016382,-0.000234,0.058876,-0.015793,-0.070493,-0.083691,0.013759,0.058123,-0.089099,0.021674


In [11]:
y

0        happy
1        happy
2        happy
3        happy
4        happy
        ...   
233    neutral
234    neutral
235    neutral
236    neutral
237    neutral
Name: class, Length: 238, dtype: object

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1234)
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')

In [13]:

from sklearn.pipeline import make_pipeline 
from sklearn.preprocessing import StandardScaler 

from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

In [14]:
pipelines = {
    'lr':make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'rc':make_pipeline(StandardScaler(), RidgeClassifier()),
    'rf':make_pipeline(StandardScaler(), RandomForestClassifier()),
    'gb':make_pipeline(StandardScaler(), GradientBoostingClassifier()),
}

In [15]:

fit_models = {}
for algo, pipeline in pipelines.items():
    model = pipeline.fit(X_train, y_train)
    fit_models[algo] = model

In [16]:
import numpy as np

print(np.isnan(X).sum())

pose_x0     0
pose_y0     0
pose_z0     0
pose_v0     0
pose_x1     0
           ..
face_y9     0
face_z9     0
face_x10    0
face_y10    0
face_z10    0
Length: 165, dtype: int64


In [17]:
fit_models

{'lr': Pipeline(steps=[('standardscaler', StandardScaler()),
                 ('logisticregression', LogisticRegression(max_iter=1000))]),
 'rc': Pipeline(steps=[('standardscaler', StandardScaler()),
                 ('ridgeclassifier', RidgeClassifier())]),
 'rf': Pipeline(steps=[('standardscaler', StandardScaler()),
                 ('randomforestclassifier', RandomForestClassifier())]),
 'gb': Pipeline(steps=[('standardscaler', StandardScaler()),
                 ('gradientboostingclassifier', GradientBoostingClassifier())])}

In [18]:
fit_models['rc'].predict(X_test)

array(['victory', 'sad', 'happy', 'sad', 'sad', 'victory', 'neutral',
       'sad', 'happy', 'neutral', 'happy', 'victory', 'neutral', 'sad',
       'neutral', 'happy', 'victory', 'happy', 'sad', 'victory', 'sad',
       'happy', 'happy', 'happy', 'victory', 'sad', 'neutral', 'happy',
       'victory', 'happy', 'sad', 'happy', 'victory', 'sad', 'neutral',
       'neutral', 'neutral', 'sad', 'neutral', 'neutral', 'victory',
       'victory', 'neutral', 'sad', 'neutral', 'neutral', 'happy',
       'happy', 'sad', 'sad', 'sad', 'neutral', 'neutral', 'sad',
       'neutral', 'neutral', 'victory', 'victory', 'victory', 'neutral',
       'victory', 'victory', 'happy', 'neutral', 'sad', 'victory',
       'victory', 'neutral', 'happy', 'victory', 'sad', 'victory'],
      dtype='<U7')

In [19]:
from sklearn.metrics import accuracy_score

for name, model in fit_models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"{name}: {acc:.4f}")

lr: 0.9861
rc: 0.9722
rf: 0.9861
gb: 0.9583


In [20]:
import pickle 

In [21]:
with open('body_language.pkl', 'wb') as f:
    pickle.dump(fit_models['rf'], f)

In [22]:
with open('body_language.pkl', 'rb') as f:
    model = pickle.load(f)

In [25]:
import cv2
import time
import numpy as np
import pickle
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# ------------------ LOAD MODEL ------------------
with open("body_language.pkl", "rb") as f:
    model = pickle.load(f)

# ------------------ LOAD MEDIAPIPE MODELS ------------------
BaseOptions = python.BaseOptions

pose_detector = vision.PoseLandmarker.create_from_options(
    vision.PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=pose_model),
        running_mode=vision.RunningMode.VIDEO
    )
)

face_detector = vision.FaceLandmarker.create_from_options(
    vision.FaceLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=face_model),
        running_mode=vision.RunningMode.VIDEO
    )
)

# ------------------ FACE POINTS ------------------
FACE_POINTS = [1, 33, 133, 362, 263, 61, 291, 13, 14, 70, 300]

# ------------------ WEBCAM ------------------
cap = cv2.VideoCapture(0)

start_time = time.time()
prev_timestamp = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

    # -------- TIMESTAMP --------
    timestamp = int((time.time() - start_time) * 1000)
    if timestamp <= prev_timestamp:
        timestamp = prev_timestamp + 1
    prev_timestamp = timestamp

    # -------- DETECTION --------
    pose_result = pose_detector.detect_for_video(mp_image, timestamp)
    face_result = face_detector.detect_for_video(mp_image, timestamp)

    try:
        # -------- POSE --------
        if pose_result.pose_landmarks:
            pose = pose_result.pose_landmarks[0]
            pose_row = np.array([[lm.x, lm.y, lm.z, lm.visibility] for lm in pose]).flatten()
        else:
            pose_row = np.zeros(33 * 4)

        # -------- FACE --------
        if face_result.face_landmarks:
            face = face_result.face_landmarks[0]
            nose = face[1]

            face_row = []
            for idx in FACE_POINTS:
                lm = face[idx]
                face_row += [
                    lm.x - nose.x,
                    lm.y - nose.y,
                    lm.z
                ]
            face_row = np.array(face_row)
        else:
            face_row = np.zeros(len(FACE_POINTS) * 3)

        # -------- CLEAN --------
        pose_row = np.nan_to_num(pose_row)
        face_row = np.nan_to_num(face_row)

        # -------- FINAL INPUT --------
        X = np.concatenate([pose_row, face_row]).reshape(1, -1).astype('float32')

        # -------- PREDICTION --------
        prediction = model.predict(X)[0]

        if hasattr(model, "predict_proba"):
            prob = np.max(model.predict_proba(X))
        else:
            prob = 0

        # -------- DISPLAY --------
        cv2.putText(frame, f"{prediction} ({prob:.2f})", (10, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)

    except Exception as e:
        pass

    cv2.imshow("Body Language Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

I0000 00:00:1776511190.262725  807114 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1776511190.314113  807119 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776511190.334236  807115 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776511190.336939  807127 face_landmarker_graph.cc:180] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
I0000 00:00:1776511190.344153  807127 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2
W0000 00:00:1776511190.345135  807128 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776511190.357710  807131 infere